# Data Quality Audit

Day 1 of the Olist Marketplace Health Audit. Before any analysis can stand, I need to confirm what the data actually contains.

Six checks across the 9 ingested tables:

1. Shape and dtypes - row counts, column counts, type breakdown.
2. Nullability - null fractions per column, with interpretation (defect vs feature).
3. Primary key uniqueness - is each table's natural PK actually unique?
4. Foreign key orphans - do child-table FKs resolve to parent rows?
5. Date range sanity - are timestamps within Olist's operating window?
6. Full-row duplicates - separate from PK checks, full dups suggest extraction bugs.

Findings summary at the end. Day 2 EDA depends on these findings.


In [1]:
import sys
from pathlib import Path

# Make src/ importable regardless of where the notebook is launched from.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.db import get_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

engine = get_engine()

TABLES = [
    "customers", "geolocation", "order_items", "order_payments",
    "order_reviews", "orders", "products", "sellers",
    "product_category_name_translation",
]

# SQLite stores datetimes as TEXT (no native datetime type), so re-parse on read.
DATE_COLS_BY_TABLE = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items":   ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

dfs = {
    t: pd.read_sql(f"SELECT * FROM {t}", engine, parse_dates=DATE_COLS_BY_TABLE.get(t))
    for t in TABLES
}

print(f"Loaded {len(dfs)} tables, {sum(len(d) for d in dfs.values()):,} total rows")


Loaded 9 tables, 1,550,922 total rows


## 1. Shape and dtypes

Per-table row count, column count, and dtype breakdown. This serves two purposes: confirming the ingest count matches the source CSVs, and confirming the date columns parsed in `src/ingest.py` arrived as `datetime64[ns]` rather than `object`.


In [2]:
def shape_and_dtypes(name: str, df: pd.DataFrame) -> dict:
    return {
        "table":     name,
        "rows":      len(df),
        "cols":      df.shape[1],
        "datetimes": df.select_dtypes(include="datetime64").shape[1],
        "numerics":  df.select_dtypes(include="number").shape[1],
        "objects":   df.select_dtypes(include="object").shape[1],
    }

shape_summary = pd.DataFrame([shape_and_dtypes(n, d) for n, d in dfs.items()])
shape_summary


,table,rows,cols,datetimes,numerics,objects
0,customers,99441,5,0,1,4
1,geolocation,1000163,5,0,3,2
2,order_items,112650,7,1,3,3
3,order_payments,103886,5,0,3,2
4,order_reviews,99224,7,2,1,4
5,orders,99441,8,5,0,3
6,products,32951,9,0,7,2
7,sellers,3095,4,0,1,3
8,product_category_name_translation,71,2,0,0,2


## 2. Nullability

Per-column null fractions. I keep only columns with any nulls (no point listing zeros). Interpretation below the table - not every null is a defect.


In [3]:
def null_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    pct = (df.isna().mean() * 100).round(2)
    pct = pct[pct > 0].sort_values(ascending=False)
    if pct.empty:
        return pd.DataFrame()
    return pd.DataFrame({"table": name, "column": pct.index, "pct_null": pct.values})

null_report = pd.concat([null_summary(n, d) for n, d in dfs.items()], ignore_index=True)
null_report


,table,column,pct_null
0,order_reviews,review_comment_title,88.34
1,order_reviews,review_comment_message,58.70
2,orders,order_delivered_customer_date,2.98
3,orders,order_delivered_carrier_date,1.79
4,orders,order_approved_at,0.16
5,products,product_category_name,1.85
6,products,product_name_lenght,1.85
7,products,product_description_lenght,1.85
8,products,product_photos_qty,1.85
9,products,product_weight_g,0.01


### Interpretation

Not every null is a defect.

- `review_comment_title` (~88%) and `review_comment_message` (~59%): most reviewers leave only a star rating. Expected.
- `orders.order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`: null when an order was cancelled or never reached that stage. The nullness itself is the signal we use on Day 2 to flag undelivered orders.
- `products.product_category_name` and dimension columns (~1.85% null): mild defect. On Day 2 when grouping by category, tag nulls as "unknown" rather than dropping, to keep row counts honest.
- `order_reviews.review_answer_timestamp`: small fraction. Some reviews never got a seller response.

I'm not fixing anything in this notebook. Documenting these is the fix; Day 2 code references them.


## 3. Primary key uniqueness

For each table, declare the natural primary key and verify it. The known Olist gotcha: `customers.customer_id` is per-order, not per-person. Cross-order identity lives in `customer_unique_id`. Missing this distinction produces wrong churn and repeat-buyer numbers.


In [4]:
PK_MAP = {
    "customers":                         ["customer_id"],
    "orders":                            ["order_id"],
    "order_items":                       ["order_id", "order_item_id"],
    "order_payments":                    ["order_id", "payment_sequential"],
    "order_reviews":                     ["review_id"],
    "products":                          ["product_id"],
    "sellers":                           ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
    # geolocation: no natural PK; multiple rows per zip prefix is by design.
}

def pk_check(name: str, keys: list[str]) -> dict:
    df = dfs[name]
    n_rows   = len(df)
    n_unique = df.drop_duplicates(subset=keys).shape[0]
    return {
        "table":     name,
        "pk":        " + ".join(keys),
        "rows":      n_rows,
        "unique_pk": n_unique,
        "is_unique": n_rows == n_unique,
    }

pk_report = pd.DataFrame([pk_check(n, k) for n, k in PK_MAP.items()])
pk_report


,table,pk,rows,unique_pk,is_unique
0,customers,customer_id,99441,99441,True
1,orders,order_id,99441,99441,True
2,order_items,order_id + order_item_id,112650,112650,True
3,order_payments,order_id + payment_sequential,103886,103886,True
4,order_reviews,review_id,99224,98410,False
5,products,product_id,32951,32951,True
6,sellers,seller_id,3095,3095,True
7,product_category_name_translation,product_category_name,71,71,True


### Verifying customer_unique_id

`customer_id` is unique above (one row per customer-order pairing). But real shoppers place multiple orders, so where does cross-order identity actually live? `customer_unique_id`.


In [5]:
customers = dfs["customers"]
n_total            = len(customers)
n_unique_customers = customers["customer_unique_id"].nunique()
n_repeat_buyers    = (customers.groupby("customer_unique_id").size() > 1).sum()

print(f"customer_id values:        {n_total:,}")
print(f"customer_unique_id values: {n_unique_customers:,}")
print(f"Customers with >1 order:   {n_repeat_buyers:,} "
      f"({n_repeat_buyers / n_unique_customers:.1%} of unique customers)")


customer_id values:        99,441
customer_unique_id values: 96,096
Customers with >1 order:   2,997 (3.1% of unique customers)


`customer_id` is per-order; `customer_unique_id` is per-person. Any per-customer metric (churn, repeat-buy, lifetime value) must aggregate by `customer_unique_id`. Day 2 EDA uses `customer_unique_id` for anything per-person.


## 4. Foreign key orphans

Every child-table FK should resolve to a row in its parent. Orphans cause silent row drops on inner joins, which is dangerous because the loss is invisible. The relationship map for this dataset is below.


In [6]:
FK_RELATIONSHIPS = [
    # (child_table, fk_col, parent_table, pk_col)
    ("orders",         "customer_id",           "customers", "customer_id"),
    ("order_items",    "order_id",              "orders",    "order_id"),
    ("order_items",    "product_id",            "products",  "product_id"),
    ("order_items",    "seller_id",             "sellers",   "seller_id"),
    ("order_payments", "order_id",              "orders",    "order_id"),
    ("order_reviews",  "order_id",              "orders",    "order_id"),
    ("products",       "product_category_name", "product_category_name_translation", "product_category_name"),
]

def orphan_check(child: str, fk: str, parent: str, pk: str) -> dict:
    child_keys    = set(dfs[child][fk].dropna().unique())
    parent_keys   = set(dfs[parent][pk].dropna().unique())
    orphans       = child_keys - parent_keys
    n_orphan_rows = int(dfs[child][fk].isin(orphans).sum()) if orphans else 0
    return {
        "child_table":  child,
        "fk_col":       fk,
        "parent_table": parent,
        "orphan_keys":  len(orphans),
        "orphan_rows":  n_orphan_rows,
    }

fk_report = pd.DataFrame([orphan_check(*rel) for rel in FK_RELATIONSHIPS])
fk_report


,child_table,fk_col,parent_table,orphan_keys,orphan_rows
0,orders,customer_id,customers,0,0
1,order_items,order_id,orders,0,0
2,order_items,product_id,products,0,0
3,order_items,seller_id,sellers,0,0
4,order_payments,order_id,orders,0,0
5,order_reviews,order_id,orders,0,0
6,products,product_category_name,product_category_name_translation,2,13


## 5. Date range sanity

Every timestamp should fall within Olist's operating window. Out-of-window dates usually indicate source-system clock errors or test data. The window I'm using (2016-01 to 2018-12) is intentionally wider than the tight operational window; the goal is to flag obvious errors, not pre-filter.


In [7]:
WINDOW_START = pd.Timestamp("2016-01-01")
WINDOW_END   = pd.Timestamp("2018-12-31")

def date_summary(table: str, col: str) -> dict:
    s = dfs[table][col].dropna()
    return {
        "table":         table,
        "column":        col,
        "min":           s.min(),
        "max":           s.max(),
        "out_of_window": int(((s < WINDOW_START) | (s > WINDOW_END)).sum()),
    }

date_report = pd.DataFrame([
    date_summary(t, c) for t, cols in DATE_COLS_BY_TABLE.items() for c in cols
])
date_report


,table,column,min,max,out_of_window
0,orders,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
1,orders,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,0
2,orders,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,0
3,orders,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,0
4,orders,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0
5,order_items,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,4
6,order_reviews,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0
7,order_reviews,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0


## 6. Full-row duplicates

A different concern from PK uniqueness. A full row duplicate is the same data reported twice, regardless of declared keys. Usual causes are upstream ETL bugs or pipelines re-running and double-loading.


In [8]:
dup_report = pd.DataFrame([
    {"table": name, "full_dup_rows": int(df.duplicated().sum())}
    for name, df in dfs.items()
])
dup_report


,table,full_dup_rows
0,customers,0
1,geolocation,261831
2,order_items,0
3,order_payments,0
4,order_reviews,0
5,orders,0
6,products,0
7,sellers,0
8,product_category_name_translation,0


## 7. Findings going into Day 2

All numbers below are pulled from the cells above. These are the assumptions every later notebook builds on.

### Identity model
- `customer_id` is per-order (99,441 unique). `customer_unique_id` is per-person (96,096 unique). Only 3.1% of customers placed more than one order. Olist is heavily one-and-done. Any per-customer aggregation (churn, repeat-buyer, lifetime value) must use `customer_unique_id`.

### review_id is not unique
- `order_reviews` has 99,224 rows but only 98,410 unique `review_id` values. 814 review_ids appear on multiple orders. Composite row identity is (`review_id`, `order_id`), never `review_id` alone.

### Nullness as signal
- `order_delivered_customer_date` (2.98% null): orders never delivered (cancelled, returned, or in-flight at extract time). Use the nullness as a feature when classifying delivery outcomes. Don't impute.
- `review_comment_title` (88.34% null) and `review_comment_message` (58.70% null): reviewer left only a star rating. Expected.

### Smaller defects to handle
- `products.product_category_name` and dimension columns are 1.85% null. On Day 2 when grouping by category, tag nulls as "unknown" rather than dropping; keeps row counts honest.
- 2 product categories have no English translation (13 product rows affected). Day-2 visualisations using English category names should LEFT JOIN to keep these rows visible.

### Big defects to design around
- `geolocation` has 261,831 full duplicate rows (about 26% of the table). Biggest finding. A naive JOIN on geolocation explodes row counts and corrupts aggregations. Day 4 (geo risk map) must collapse `geolocation` to one row per zip prefix (median lat/lng) before joining.
- `order_items.shipping_limit_date` has 4 rows beyond the operational window (max 2020-04-09, about 1.5 years past the stated dataset end). Source-clock errors. Exclude from any time-series shipping-SLA analysis.

### Operating window
- Real `order_purchase_timestamp` window: 2016-09-04 to 2018-10-17. Filter to this range when computing rates over time.

### Joins
- 6 of 7 documented FK relationships are clean (zero orphans).
- The exception is `products.product_category_name` to `product_category_name_translation` (2 orphan keys, 13 rows). Use LEFT JOIN.

### Day 2 implications
- Use `customer_unique_id` for any per-customer aggregation.
- Treat null delivery dates as the "undelivered" feature.
- Collapse `geolocation` (median lat/lng per zip prefix) before joining.
- Tag null product categories as "unknown".
- Filter to 2016-09 to 2018-10 for any time-windowed metric.
